# Ollama + OpenAI + Python

## 1. បញ្ជាក់ឈ្មោះម៉ូដែល

ប្រសិនបើអ្នកបានយកម៉ូដែលផ្សេងពី "phi3:mini", សូមផ្លាស់ប្តូរតម្លៃនៅក្នុងកោសិ៍ខាងក្រោម.  
អថេរនោះនឹងត្រូវបានប្រើក្នុងកូដទូទាំងសៀវភៅកំណត់ត្រា.


In [ ]:
MODEL_NAME = "phi3:mini"

## 2. ការតំឡើង Open AI client

ជាធម្មតា client នៃ OpenAI ត្រូវបានប្រើជាមួយ OpenAI.com ឬ Azure OpenAI ដើម្បីធ្វើអន្តរកម្មជាមួយម៉ូឌែលភាសាធំៗ។
ទោះជាយ៉ាងណា វាក៏អាចប្រើជាមួយ Ollama បានផងដែរ ពីព្រោះ Ollama ផ្តល់នូវ endpoint ដែលផ្គូផ្គងជាមួយ OpenAI នៅ "http://localhost:11434/v1".


In [ ]:
%pip install openai

In [ ]:
import openai

client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="nokeyneeded",
)

## 3. បង្កើតការបញ្ចប់សន្ទនា

ឥឡូវនេះ យើងអាចប្រើ OpenAI SDK ដើម្បីបង្កើតចម្លើយសម្រាប់សន្ទនា។ សំណើនេះគួរបង្កើត haiku អំពីឆ្មា:


In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.7,
    n=1,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about a hungry cat"},
    ],
)

print("Response:")
print(response.choices[0].message.content)


## 4. ការ​រចនា​សំណើ

សារ​ដំបូង​ដែល​ផ្ញើ​ទៅ​ម៉ូឌែល​ភាសា ត្រូវ​បាន​ហៅ​ថា "system message" ឬ "system prompt" ហើយ​វា​កំណត់​ណែនាំ​ទូទៅ​សម្រាប់​ម៉ូឌែល។
អ្នក​អាច​ផ្តល់ system prompt ផ្ទាល់ខ្លួន ដើម្បីណែនាំ​ម៉ូឌែល​ភាសា​ឱ្យ​បង្កើត​លទ្ធផល​ក្នុង​របៀប​ផ្សេងៗ។
កែ​ឡើង `SYSTEM_MESSAGE` ខាងក្រោម ដើម្បីឆ្លើយ​ដូច​ជា​តួអង្គ​ភាគច្រើន​ដែលអ្នក​ចូលចិត្ត ក្នុងភាពយន្ដ/ទូរទស្សន៍ ឬ ទទួល​បាន​ស្នាមគំនិត​សម្រាប់ system prompts ផ្សេងៗ​ពី [Awesome ChatGPT Prompts](https://github.com/f/awesome-chatgpt-prompts?tab=readme-ov-file#prompts)។

ពេល​ដែល​អ្នក​បានប្ដូរ system message ហើយ សូម​ផ្តល់​សំណួរ​ប្រើប្រាស់​ដំបូង​នៅ​ក្នុង `USER_MESSAGE`។


In [ ]:
SYSTEM_MESSAGE = """
I want you to act like Elmo from Sesame Street.
I want you to respond and answer like Elmo using the tone, manner and vocabulary that Elmo would use.
Do not write any explanations. Only answer like Elmo.
You must know all of the knowledge of Elmo, and nothing more.
"""

USER_MESSAGE = """
Hi Elmo, how are you doing today?
"""

response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.7,
    n=1,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print("Response:")
print(response.choices[0].message.content)


## 5. ឧទាហរណ៍ few-shot

វិធី​មួយ​ផ្សេង​ទៀត​ក្នុង​ការ​ណែនាំ​ម៉ូឌែល​ភាសា​គឺ​ផ្តល់ "few shots", ស៊េរី​នៃ​ឧទាហរណ៍​សំណួរ/ចម្លើយ​ដែល​បង្ហាញ​ពី​របៀប​ដែល​វា​គួរ​ឆ្លើយ​តប។

ឧទាហរណ៍​ខាង​ក្រោម​ព្យាយាម​ធ្វើ​ឲ្យ​ម៉ូឌែល​ភាសា​ធ្វើ​ដូចជា​ជំនួយការ​បង្រៀន ដោយ​ផ្តល់ឧទាហរណ៍​សំណួរ និង​ចម្លើយ​ពី​របី​ដែល TA អាច​ផ្តល់​បាន ហើយ​បន្ទាប់​មក​ស្នើ​ម៉ូឌែល​ជាមួយ​សំណួរ​មួយ​ដែល​សិស្ស​អាច​សួរ។

សាកល្បង​វា​មុន ហើយ​បន្ទាប់​មក​កែសម្រួល `SYSTEM_MESSAGE`, `EXAMPLES`, និង `USER_MESSAGE` សម្រាប់​សេណារីយ៉ូ​ថ្មី។


In [ ]:
SYSTEM_MESSAGE = """
You are a helpful assistant that helps students with their homework.
Instead of providing the full answer, you respond with a hint or a clue.
"""

EXAMPLES = [
    (
        "What is the capital of France?",
        "Can you remember the name of the city that is known for the Eiffel Tower?"
    ),
    (
        "What is the square root of 144?",
        "What number multiplied by itself equals 144?"
    ),
    (   "What is the atomic number of oxygen?",
        "How many protons does an oxygen atom have?"
    ),
]

USER_MESSAGE = "What is the largest planet in our solar system?"


response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.7,
    n=1,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": EXAMPLES[0][0]},
        {"role": "assistant", "content": EXAMPLES[0][1]},
        {"role": "user", "content": EXAMPLES[1][0]},
        {"role": "assistant", "content": EXAMPLES[1][1]},
        {"role": "user", "content": EXAMPLES[2][0]},
        {"role": "assistant", "content": EXAMPLES[2][1]},
        {"role": "user", "content": USER_MESSAGE},
    ],
)


print("Response:")
print(response.choices[0].message.content)

## 6. ការបង្កើតពង្រឹងដោយការស្វែងយក

RAG (Retrieval Augmented Generation) គឺជាវិធីសាស្ត្រមួយដើម្បីធ្វើឱ្យម៉ូឌែលភាសាចម្លើយសំណួរបានយ៉ាងត្រឹមត្រូវសម្រាប់ដែនកំណត់ពិសេសមួយ ដោយចាប់ផ្តើមពីការទាញយកព័ត៌មានដែលទាក់ទងពីប្រភពចំណេះដឹង ម្តងហើយបន្ទាប់មកបង្កើតចម្លើយនៅលើមូលដ្ឋានព័ត៌មាននោះ។

យើងបានផ្តល់ឯកសារ CSV ក្នុងស្រុកដែលមានទិន្នន័យអំពីរថយន្តហាយប៊្រីដ។ កូដនៅខាងក្រោមអានឯកសារ CSV នោះ ស្វែងរកការផ្គូប្រកួតទៅនឹងសំនួរអ្នកប្រើ ហើយបន្ទាប់មកបង្កើតចម្លើយដោយយោងទៅលើព័ត៌មានដែលបានឃើញ។ សូមចាប់អារម្មណ៍ថា នេះនឹងចំណាយពេលយូរជាងឧទាហរណ៍មុនៗទាំងអស់ ព្រោះវាបញ្ជូនទិន្នន័យច្រើនទៅកាន់ម៉ូឌែល។ ប្រសិនបើអ្នកឃើញថាចម្លើយនៅតែមិនផ្អែកលើទិន្នន័យ អ្នកអាចព្យាយាមធ្វើវិស្វកម្មប្រព័ន្ធ ឬសាកល្បងម៉ូឌែលផ្សេងៗ។ ទូទៅ RAG មានប្រសិទ្ធភាពបន្ថែមពេលប្រើម៉ូឌែលធំ ឬជាមួយកំណែដែលបានកែតម្រូវយ៉ាងម៉ត់ចត់នៃ SLMs។


In [ ]:
import csv

SYSTEM_MESSAGE = """
You are a helpful assistant that answers questions about cars based off a hybrid car data set.
You must use the data set to answer the questions, you should not provide any information that is not in the provided sources.
"""

USER_MESSAGE = "how fast is a prius?"

# Open the CSV and store in a list
with open("hybrid.csv", "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

# Normalize the user question to replace punctuation and make lowercase
normalized_message = USER_MESSAGE.lower().replace("?", "").replace("(", " ").replace(")", " ")

# Search the CSV for user question using very naive search
words = normalized_message.split()
matches = []
for row in rows[1:]:
    # if the word matches any word in row, add the row to the matches
    if any(word in row[0].lower().split() for word in words) or any(word in row[5].lower().split() for word in words):
        matches.append(row)

# Format as a markdown table, since language models understand markdown
matches_table = " | ".join(rows[0]) + "\n" + " | ".join(" --- " for _ in range(len(rows[0]))) + "\n"
matches_table += "\n".join(" | ".join(row) for row in matches)
print(f"Found {len(matches)} matches:")
print(matches_table)

# Now we can use the matches to generate a response
response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.7,
    n=1,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_MESSAGE + "\nSources: " + matches_table},
    ],
)

print("Response:")
print(response.choices[0].message.content)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្ម​បកប្រែដោយ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ថ្វីទាំងយើងខំប្រឹងដើម្បីភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថា ការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានយកជា​ប្រភពដែលមានសុពលភាព។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងណែនាំឱ្យប្រើការបកប្រែដោយអ្នកប្រែវិជ្ជាជីវៈមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយ ដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
